# Chicago weather forecast evaluation

Compare Open-Meteo forecasts made 24 and 48 hours before each valid hour against normalized historical weather. Production metric logic lives in `models/weather/evaluate_forecast_skill.py`; this notebook is for inspection and reporting.

In [ ]:
from pathlib import Path

import pandas as pd

from models.weather.evaluate_forecast_skill import (
    align_forecasts_with_observations,
    evaluate_forecast_skill,
)

## Select matching files

Both files must cover the same valid-time range. Change these paths to the range downloaded locally.

In [ ]:
FORECAST_FILE = Path(
    "data/processed/weather/forecast_evaluation/"
    "open_meteo_previous_runs_chicago_2024-01-01_2024-12-31.parquet"
)
OBSERVED_FILE = Path(
    "data/processed/weather/"
    "open_meteo_chicago_2024-01-01_2024-12-31.parquet"
)

forecasts = pd.read_parquet(FORECAST_FILE)
observations = pd.read_parquet(OBSERVED_FILE)

print(f"Forecast rows: {len(forecasts):,}")
print(f"Observed rows: {len(observations):,}")

In [ ]:
aligned = align_forecasts_with_observations(forecasts, observations)
aligned.groupby("lead_time_hours")["timestamp_utc"].agg(
    ["count", "min", "max"]
)

## Skill by forecast horizon

Rain is counted at more than 0.1 mm per hour and snow at more than 0.1 cm per hour. Adjust thresholds only when the choice is documented.

In [ ]:
metrics = evaluate_forecast_skill(
    forecasts,
    observations,
    rain_threshold_mm=0.1,
    snow_threshold_cm=0.1,
)
metrics

In [ ]:
METRICS_FILE = Path(
    "data/processed/weather/forecast_evaluation/"
    "weather_forecast_skill_2024.csv"
)
METRICS_FILE.parent.mkdir(parents=True, exist_ok=True)
metrics.to_csv(METRICS_FILE, index=False)
METRICS_FILE